# An introduction to Jupyter notebooks for Crossref

Jupyter notebooks can help to share results from API queries without needing a lot of technical knowledge, or enable rerunning a customized report for different members or at different points in time. Here we'll explore a few things:

1. How to open, run, and share a notebook.
1. A few basic commands to get started.

We won't cover learning and installing Python and Jupyter notebooks, but you can use Colab from Google Drive or search for a guide to installing Python on your computer. You should be able to run through this notebook without any previous experience of programming.

## What is a Notebook?

A Jupyter notebook is a way to put text and code together. It's mainly designed to run Python code, but other languages, and images can also be used. You can use a Notebook for:

1. Sharing an API query with someone for them to run themself.
1. An asynchronous tutorial or demonstration (like this one!).
1. Creating a report that is run on a regular basis.
1. Experimenting and viewing output as you go along.

There are a few instances that a notebook might not be the best solution, e.g.:
1. Creating a function to use in multiple different files. Instead, save it as a python script and call it each time, or create a library that can be loaded.
1. Writing and sharing complex code with developers or others familiar with Python.
1. Handling very large datasets or outputs. There is a limit to how much can be displayed and with large data it can be more
efficient to use a Python script.

## Getting started

If you're viewing this Notebook on Colab, if you want to save changes select 'Save a copy in Drive' from the 'File' menu above.

## Entering text

Double-click on this text to see how text is formatted. It uses Markdown, in which certain characters indicate what the output should look like. For example, here's a list:

1. Item 1
1. Item 2
1. Item 3

Or you can use bullets

 - Item 1
 - Item 2
 - Item 3

You can enter code, i.e. monospaced text `like this`, or use *Bold* and _italic_ formatting.

Once you've read through this text in Markdown format, press shift-enter on your keyboard or click the arrow at the top left of this cell to execute it and it will turn into the formatted version.

Now it's your turn. Double click on 'Add your text here!' to create some text of your own.

Add your text here!

# Running Python

We're not going to cover Python in detail here and the great thing about a notebook is that you can execute code without needing to fully understand it. The text alongside should explain what a function will do and what output you will expect.

Code is entered in a cell and the output appears below. With Python you can do everything from simple arithmetic to building websites and querying APIs.

Click on the next cell, then execute it (using shift+enter or clicking on the arrow) to see the result.

In [ ]:
8 + 5

Congratulations, you've just run some Python!

Python uses different types of objects. The next cell shows a few common examples that you'll see used in the rest of the notebook. To assign an object to a variable use `=`.

In [ ]:
# Integers
example_int = 8

# Float (numbers with a decimal point)
example_float = 3.142

# string (text)
example_str = 'Would you like a cup of tea?'

# List (note that the items in the list can have different types)
example_list = [5, 'cup of tea', ['list', 'in', 'a', 'list']]

# Dictionary (each entry has a key, which is a string, and a value, much like json)
example_dict = {
    'key1': 'value1',
    'key2': 'value2'
}


The print statement allows us to show outupt to the user.

---



In [ ]:
print(example_int)
print(f'example_int has type {type(example_int)}')

Change the content of the print statements to show information about one of the other variables, e.g. `example_str`

Python (like most other computer languages) makes extensive use of libraries which contain code that's already been written and you can reuse. We're going to look at three here that are useful for querying APIs:

 - `requests` to query APIs
 - `json` to load and save json
 - `pprint` to format output nicely

Let's start with importing them. Execute the next cell. You won't see any output, but it will enable some functions to use later.

In [ ]:
# lines with a hash symbol before are a comment and aren't executed
# they can be used to make notes and add explanantions

# let's load some libraries
import requests
import json
import pprint

## Make an API query

APIs contain data that is formatted in a way that makes it easy for computer programs to interact with. Let's see it in action.

We'll query Crossref's /works API endpoint, and save and view the output. We're going to get:

 - 5 metadata records (`'rows': 5`)
 - only journal articles (`type:journal-article`)
 - works published between two dates (`from-pub-date:2024-01-01,until-pub-date:2024-01-01`)

In [ ]:
# the API address
url = 'https://api.crossref.org/v1/works'

# Parameters for the query.
# These go into a python object called a dictionary which has keys (e.g. 'rows') and values (e.g. 10).
query_parameters = {
    'rows':5,
    'filter': 'type:journal-article,from-pub-date:2024-01-01,until-pub-date:2024-01-01'
}

# let's make the query
r = requests.get(url, params=query_parameters)

So where are the results? They're saved in the object `r` which the query returned and we can use below. If there's json in the result we can get it by calling the function `r.json()`.

There's a useful value `r.status_code` which tells will have a value of 200 if the query was successful. Let's do a check to see if the query ran successfully and if it didn't we'll print out `r.text` which should contain an error message.

In [ ]:
print(f"The status code is: {r.status_code}")
if r.status_code == 200:
    js = r.json()
else:
    print(r.text)

Assuming all went well, let's look at the json. `pprint.pprint` formats a Python dictionary nicely (if you like, compare it with the native command `print(js)`).

In [ ]:
pprint.pprint(js)

We can't see all of the output, but at least we can tell that it's more or less the right thing.

We can get some specific information from the output by using the field names in the JSON (which correspond to keys in a Python dictionary), for example to get the total number of results:

In [ ]:
total_results = js['message']['total-results']

# use f"..." and curly brackets to combine variables and text
print(f"{total_results} metadata records were found")

Let's save it as a json file. If you're using Colab, the file will appear in the window on the left. Click the folder icon to view the files.

Note that these files are deleted if you restart the notebook so be sure to download the files or more them to your Google drive if you want to keep them!

In [ ]:
# this is what your file will be called
filename = 'crossref.json'

# this is how you save files using json
# if the file already exists it will be overwritten!
with open(filename, 'w') as f:
    json.dump(js, f, indent=2)

Double click on 'crossref.json' in the files on the left to open it and view the output. It should show up in a window on the right.

## Functions: Doing it all in one go

A Python function can make it much more efficient to run something multiple times. Here's a function that combines some of the lines above to query the Crossref API. We've also added a subfunction that deals with the weirdness of filters in the works endpoint, and another to save something to a json file.

This looks like a lot of code, but when we want to use it we can do it in just one line.

Note that we can explain what we're doing as we go along using comments. Anything in Python with `#` before it, or between triple quotation marks (`'''...'''`) is ignore when running the code.

In [ ]:
works_url = 'https://api.crossref.org/v1/works'

def get_works(params:dict, filters = {}) -> tuple[dict, int]:
    ''' Query the Crossref works endpoint and return json '''

    # reformat and add filters to the query parameters, if used
    if len(list(filters.keys())) > 0:
        params['filter'] = filters_to_params(filters)

    # make the query
    r = requests.get(works_url, params=params)

    # get the json output from the API response
    if r.status_code == 200:
        js = r.json()
    else:
        # something went wrong, print the response and return an empty dictionary
        print(r.text)
        js = {}
    return js, r.status_code


def filters_to_params(filters:dict) -> str:
    ''' Set the filters parameter for the Crossref works API using a dictionary '''

    # create an empty string for the filter values
    fval = ''
    # add each filter key and value to the string
    for f in filters:
        fval += str(f) + ':' + str(filters[f]) + ','
    fval = fval[:-1] # removing trailing comma

    return fval

def save_to_json(data:dict, fname:str) -> None:
    with open(fname, 'w') as f:
        json.dump(data, f, indent=2)

Let's run a query using one of these functions. Notice how much shorter this is than the previous cell!

Unlike the above, we can put the filters into a separate dictionary and our function does the rest for us. Here we get:

 - 5 metadata records (`'rows':5`)
 - works published after 1 October 2023 (`'from-pub-date': '2023-10-01'`)
 - updates, i.e. corrections, retractions, errata (`'is-update': 1`)

We've got the json output back, let's see if it looks as we expect:

In [ ]:
# set up the parameters
query_params = {
    'rows': 5
}

# put the filters into another dictionary
filters = {
    'from-pub-date': '2023-10-01',
    'is-update': 1
}

# let's run the function we defined above
js, status = get_works(query_params, filters)

In [ ]:
pprint.pprint(js)

In [ ]:
# now save it to a file in a single line
save_to_json(js, 'crossref.json')

Now it's your turn! Change the query parameters or filters above. For example:
 - Add a mailto parameter with your email address.
 - Get works with an ORCID and/or funding.

 You can see all of the posssible parameters and filters in the [API documentation](http://api.crossref.org/swagger-ui/index.html#/Works/get_works)

# Tips and tricks

Here are a few tips for dealing with notebooks.

### 1. Copy and paste

Use other people's code! Copy the functions and things from above here. If it works well and gives the correct output it's worth using again and again. Also make use of libraries to shortcut your code.

### 2. Plan the structure

Before you write any code, write a short plan of what you will do. Think especially about the outputs you're expecting and what format they will be in. Do you need extra steps to clean up the data, or some analysis to pick out its most important features? How will you share the results? Which libraries and functions do you need?

### 3. Don't do too much at once

Keep your code cells short and split up different steps, at least until you're confident they're working. That helps to identify where problems are. If your analysis is complex, try first with something very simple (like just a few rows of data or a single member) before you scale up.

### 4. Don't repeat yourself

If you find you're entering the same series of commands again and again, see if you can put them into a function.

### 5. Get advice

It's easy to struggle for a long time to do something simple - you usually just need the right command, or you might have missed a single character (like a bracket!)). There are plenty of forums online and colleagues who would be very happy to give advice.